# Part II - Ford GoBike Trip Duration Analysis
## by Saumyadeep

## Investigation Overview

The goal of this presentation is to communicate key findings about what influences trip duration in the Ford GoBike bike-sharing system (February 2019, San Francisco Bay Area).

## Dataset Overview and Executive Summary

The dataset contains **183,412 bike trips** from February 2019 with 16 features including trip duration, timestamps, station info, user type, and member demographics.

**Key Insights:**

1. **Trip duration is heavily right-skewed** — most trips are 5–15 minutes (commutes), but a long tail of leisure rides exists. Median ~9 min, mean ~12 min.
2. **Subscribers ride shorter than Customers** — Subscribers (annual members) have a median ~8 min trip; Customers (casual/tourist) have ~15 min. Subscribers are commuters; Customers are explorers.
3. **Usage peaks at commute hours (8–9 AM, 5–6 PM) on weekdays** — confirming the commuter-dominated system. Weekends show flatter, midday patterns with longer average durations.
4. **Younger riders (20–35) ride slightly shorter on average** than older riders — likely because younger users are more often Subscribers doing quick commutes.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

import warnings
warnings.simplefilter("ignore")

In [2]:
# Load dataset and perform wrangling (same as Part I)
df = pd.read_csv('201902-fordgobike-tripdata (1).csv')

df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])

df['duration_min'] = df['duration_sec'] / 60
df['member_age'] = 2019 - df['member_birth_year']
df['hour'] = df['start_time'].dt.hour
df['day_of_week'] = df['start_time'].dt.day_name()
df['day_type'] = np.where(df['day_of_week'].isin(['Saturday', 'Sunday']),
                          'Weekend', 'Weekday')

# Cleaning: drop trips > 60 min and implausible ages.
# NaN ages are KEPT — member_age is missing for 16.29% of Customer trips vs 3.07%
# of Subscriber trips, so dropping them would bias every user-type comparison.
df_clean = df[(df['duration_min'] <= 60)
              & ((df['member_age'].between(16, 70)) | df['member_age'].isna())].copy()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_clean['day_of_week'] = pd.Categorical(df_clean['day_of_week'],
                                         categories=day_order, ordered=True)

# Audit the cleaning — confirm it did not remove one user type preferentially.
print(f"Raw trips:      {len(df):,}")
print(f"After cleaning: {len(df_clean):,}  ({len(df) - len(df_clean):,} removed)")
print(f"  duration > 60 min:  {(df['duration_min'] > 60).sum():,}")
print(f"  age outside 16-70:  {(~df['member_age'].between(16, 70) & df['member_age'].notna()).sum():,}")
print(f"  NaN ages retained:  {df_clean['member_age'].isna().sum():,}")
print("\nRetention rate by user type:")
print((df_clean['user_type'].value_counts() / df['user_type'].value_counts() * 100).round(2))

In [ ]:
raw = df.groupby('user_type')['duration_min'].agg(n='size', median='median', mean='mean')
cln = df_clean.groupby('user_type')['duration_min'].agg(n='size', median='median', mean='mean')
print("Uncapped (all trips):\n", raw.round(2))
print("\nCapped at 60 min:\n", cln.round(2))
print(f"\nMedian gap uncapped: {raw.loc['Customer','median'] - raw.loc['Subscriber','median']:.2f} min")
print(f"Median gap capped:   {cln.loc['Customer','median'] - cln.loc['Subscriber','median']:.2f} min")

## Visualization 1: Distribution of Trip Duration

> Most trips are short commutes. The distribution is right-skewed, with the majority of trips lasting between 5 and 15 minutes. This confirms that the Ford GoBike system is primarily used for short-distance urban transportation rather than long leisure rides.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_clean['duration_min'], bins=np.arange(0, 62, 2),
        color='steelblue', edgecolor='white', alpha=0.85)

median_dur = df_clean['duration_min'].median()
mean_dur = df_clean['duration_min'].mean()
pct_under20 = (df_clean['duration_min'] < 20).mean() * 100
pct_capped = (df['duration_min'] > 60).mean() * 100

ax.axvline(median_dur, color='red', ls='--', lw=2,
           label=f'Median: {median_dur:.1f} min')
ax.axvline(mean_dur, color='darkorange', ls='--', lw=2,
           label=f'Mean: {mean_dur:.1f} min')

ax.set_title('Most Ford GoBike Trips Are Short: Half Finish Within '
             f'{median_dur:.0f} Minutes', fontsize=14, fontweight='bold')
ax.set_xlabel('Trip Duration (minutes)', fontsize=12)
ax.set_ylabel('Number of Trips', fontsize=12)
ax.set_xlim(0, 60)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.legend(fontsize=11)

ax.annotate(f'{pct_under20:.1f}% of trips finish in under 20 minutes',
            xy=(20, ax.get_ylim()[1] * 0.72), fontsize=11, color='#333')
ax.text(0.99, -0.13,
        f'Trips over 60 minutes ({pct_capped:.2f}% of all trips) are excluded; '
        'they reflect undocked bikes rather than rides.',
        transform=ax.transAxes, ha='right', fontsize=9, color='grey')

plt.tight_layout()
plt.show()

## Visualization 2: Trip Duration by User Type (Subscriber vs Customer)

> Subscribers (annual members) take significantly shorter trips than Customers (casual users). This supports the hypothesis that Subscribers are daily commuters making quick point-to-point trips, while Customers are tourists or occasional riders exploring the city at a leisurely pace.

In [ ]:
# Visualization 2: Trip Duration by User Type (box plot + mean comparison)
USER_ORDER = ['Customer', 'Subscriber']
COLORS = {'Customer': '#FF9800', 'Subscriber': '#2196F3'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: distribution
sns.boxplot(data=df_clean, x='user_type', y='duration_min',
            order=USER_ORDER, hue='user_type', hue_order=USER_ORDER,
            palette=COLORS, legend=False, ax=axes[0],
            showfliers=False, showmeans=True,
            meanprops=dict(marker='D', markerfacecolor='black',
                           markeredgecolor='white', markersize=7),
            medianprops=dict(color='red', lw=2))
axes[0].set_title('Customers Ride 55% Longer Than Subscribers',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('User Type', fontsize=12)
axes[0].set_ylabel('Trip Duration (minutes)', fontsize=12)

stats = df_clean.groupby('user_type')['duration_min'].agg(['median', 'mean', 'count'])
for i, ut in enumerate(USER_ORDER):                     # same list as order=
    axes[0].annotate(f"median {stats.loc[ut, 'median']:.2f}\nmean   {stats.loc[ut, 'mean']:.2f}",
                     xy=(i, stats.loc[ut, 'median']), xytext=(i, 32),
                     ha='center', fontsize=10, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='grey', alpha=0.9))
axes[0].set_ylim(0, 36)
axes[0].text(0.5, -0.13, 'Outlier points hidden; whiskers span 1.5 x IQR.',
             transform=axes[0].transAxes, ha='center', fontsize=9, color='grey')

# Right: mean and volume
means = [stats.loc[ut, 'mean'] for ut in USER_ORDER]
bars = axes[1].bar(USER_ORDER, means,
                   color=[COLORS[ut] for ut in USER_ORDER],
                   edgecolor='black', alpha=0.85)
axes[1].set_title('Average Duration and Trip Volume', fontsize=13, fontweight='bold')
axes[1].set_xlabel('User Type', fontsize=12)
axes[1].set_ylabel('Average Duration (minutes)', fontsize=12)
for bar, ut in zip(bars, USER_ORDER):
    n = int(stats.loc[ut, 'count'])
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"{stats.loc[ut, 'mean']:.2f} min\n({n:,} trips, "
                 f"{n / len(df_clean) * 100:.1f}%)",
                 ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylim(0, max(means) * 1.3)

plt.tight_layout()
plt.show()

> Customers ride substantially longer than Subscribers: a median of 12.65 minutes against
8.15, a gap of 4.50 minutes. Put the other way, Customers ride about **55% longer**, and
the means agree at 15.22 against 9.84 minutes.

> Subscribers account for **89.6% of all trips**. Note this is a share of *trips*, not of
riders — the dataset carries no user identifier, so nothing here can be said about how
many distinct people are in each group.

> Mean exceeds median in both groups (by a factor of 1.20 for Customers and 1.21 for
Subscribers), so both distributions are right-skewed to a similar degree. The skew is a
property of bike-share trips generally rather than of either user type.

> The two IQRs overlap by about 4.2 minutes, and the Customer 25th percentile sits almost
exactly at the Subscriber median — roughly a quarter of Customer trips are shorter than
the typical Subscriber trip. User type shifts the distribution reliably; it does not
separate riders into two distinct populations.

## Visualization 3: Trip Patterns by Hour and Day of Week


In [ ]:
USER_ORDER = ['Customer', 'Subscriber']
COLORS = {'Customer': '#FF9800', 'Subscriber': '#2196F3'}

grouped = (df_clean.groupby(['day_of_week', 'user_type'], observed=True)
                   .size().unstack(fill_value=0))
grouped.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

ax = grouped[USER_ORDER].plot(kind='bar', figsize=(12, 6), rot=0,
                              color=[COLORS[c] for c in USER_ORDER],
                              edgecolor='black', width=0.75)
ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Number of Trips', fontsize=12)
ax.set_title('Subscribers Drop Sharply at Weekends; Customers Hold Steady',
             fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.legend(title='User Type')
plt.tight_layout()
plt.show()

print(grouped[USER_ORDER].to_string())

In [ ]:
# Hour x Day trip-volume heatmap — shows actual trip counts, not summary statistics
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

heat = (df_clean.pivot_table(index='day_of_week', columns='hour',
                             values='duration_min', aggfunc='size',
                             observed=True)
                .reindex(day_order)
                .fillna(0))
heat.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, ax = plt.subplots(figsize=(14, 5.5))
sns.heatmap(heat, cmap='YlOrRd', linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Number of Trips'}, ax=ax)
ax.set_title('Weekday Commute Peaks at 8 AM and 5 PM Vanish at Weekends',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)      # no angled labels
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

peak = heat.stack().idxmax()
print(f"Busiest slot: {peak[0]} at {peak[1]}:00 with {int(heat.stack().max()):,} trips")
print("\nWeekday vs weekend trips per day, by hour band:")
band = pd.cut(df_clean['hour'], [-1, 5, 9, 15, 19, 23],
              labels=['00-05', '06-09', '10-15', '16-19', '20-23'])
per_day = (df_clean.groupby(['day_type', band], observed=True).size()
                   .unstack(0) / pd.Series({'Weekday': 20, 'Weekend': 8}))
print(per_day.round(0).to_string())

In [ ]:
# Visualization 2: Trip Duration by User Type
# Panels 1-2 are summary statistics; panel 3 shows the actual distributions,
# as required when presenting summarised values.
from matplotlib.ticker import FuncFormatter

USER_ORDER = ['Customer', 'Subscriber']
COLORS = {'Customer': '#FF9800', 'Subscriber': '#2196F3'}

stats = (df_clean.groupby('user_type')['duration_min']
                 .agg(n='size', mean='mean', median='median',
                      q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75)))

fig, axes = plt.subplots(1, 3, figsize=(19, 5.8))

# --- Panel 1 (summary): box plot ---------------------------------------------
sns.boxplot(data=df_clean, x='user_type', y='duration_min',
            order=USER_ORDER, hue='user_type', hue_order=USER_ORDER,
            palette=COLORS, legend=False, ax=axes[0],
            showfliers=False, showmeans=True,
            meanprops=dict(marker='D', markerfacecolor='black',
                           markeredgecolor='white', markersize=7),
            medianprops=dict(color='red', lw=2))
axes[0].set_title('Summary: Quartiles and Centre', fontsize=12, fontweight='bold')
axes[0].set_xlabel('User Type', fontsize=11)
axes[0].set_ylabel('Trip Duration (minutes)', fontsize=11)
axes[0].set_ylim(0, 36)
for i, ut in enumerate(USER_ORDER):                       # same list as order=
    axes[0].annotate(f"median {stats.loc[ut, 'median']:.2f}\nmean   {stats.loc[ut, 'mean']:.2f}",
                     xy=(i, 32.5), ha='center', fontsize=9.5, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='grey'))
axes[0].legend(handles=[Line2D([0], [0], color='red', lw=2, label='Median'),
                        Line2D([0], [0], lw=0, marker='D', markerfacecolor='black',
                               markeredgecolor='white', markersize=7, label='Mean')],
               fontsize=9, loc='lower right')

# --- Panel 2 (summary): mean duration and volume ------------------------------
means = [stats.loc[ut, 'mean'] for ut in USER_ORDER]
bars = axes[1].bar(USER_ORDER, means, color=[COLORS[ut] for ut in USER_ORDER],
                   edgecolor='black', width=0.6)
axes[1].set_title('Summary: Mean Duration and Trip Volume', fontsize=12, fontweight='bold')
axes[1].set_xlabel('User Type', fontsize=11)
axes[1].set_ylabel('Mean Duration (minutes)', fontsize=11)
axes[1].set_ylim(0, max(means) * 1.35)
for bar, ut in zip(bars, USER_ORDER):
    n = int(stats.loc[ut, 'n'])
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.35,
                 f"{stats.loc[ut, 'mean']:.2f} min\n{n:,} trips ({n / len(df_clean) * 100:.1f}%)",
                 ha='center', fontsize=9.5, fontweight='bold')

# --- Panel 3 (actual data): full distributions --------------------------------
for ut in USER_ORDER:
    axes[2].hist(df_clean.loc[df_clean['user_type'] == ut, 'duration_min'],
                 bins=np.arange(0, 62, 2), density=True,
                 histtype='step', lw=2.4, color=COLORS[ut], label=ut)
    axes[2].axvline(stats.loc[ut, 'median'], color=COLORS[ut], ls='--', lw=1.4)
axes[2].set_title('Actual Data: Underlying Distributions', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Trip Duration (minutes)', fontsize=11)
axes[2].set_ylabel('Density', fontsize=11)
axes[2].set_xlim(0, 60)
axes[2].legend(title='User Type', fontsize=10)

fig.suptitle('Customers Ride 55% Longer Than Subscribers — But the Distributions Overlap Heavily',
             fontsize=14, fontweight='bold', y=1.02)
fig.text(0.99, -0.02,
         'Outlier points hidden in panel 1; panel 3 is density-normalised so the 8.6:1 '
         'volume imbalance does not hide the Customer curve. Dashed lines mark medians.',
         ha='right', fontsize=9, color='grey')
plt.tight_layout()
plt.show()

print(stats.round(2).to_string())

## Investigation Overview

This presentation communicates what actually drives trip duration in the Ford GoBike
system (February 2019, San Francisco Bay Area). The short answer is that **user type
explains far more than any demographic feature**, and that duration and volume respond to
weekends in opposite directions.

## Dataset Overview

The raw dataset contains **183,412 trips** across 16 features: trip duration, timestamps,
station identifiers and coordinates, bike ID, user type, and member demographics.
**181,130 trips (98.8%)** remain after removing rides over 60 minutes and ages outside
16–70. Rows with missing demographics are deliberately retained.

## Key Insights

**1. Trip duration is short and right-skewed.** The median trip is 8.2 minutes against a
mean of 10.4, and roughly 92% of trips finish within 20 minutes. Mean exceeds median by a
factor of about 1.2 in every subgroup examined, so the skew is a property of bike-share
trips generally rather than of any particular rider group.

**2. User type is the dominant driver — Customers ride 55% longer.** Median duration is
12.65 minutes for Customers against 8.15 for Subscribers, a gap of 4.50 minutes; the means
agree at 15.22 and 9.84. Subscribers account for 89.6% of all trips (a share of *trips* —
the data carries no user identifier, so no claim about rider counts is possible).

**3. The gap is real but the distributions overlap heavily.** The two interquartile ranges
share about 4.2 minutes, and the Customer 25th percentile (8.18 min) sits almost exactly
at the Subscriber median (8.15). Roughly a quarter of Customer trips are shorter than the
typical Subscriber trip. User type reliably *shifts* the duration distribution; it does not
split riders into separable populations.

**4. Demographics add almost nothing.** Age explains **0.07% of the variance** in trip
duration (Pearson r = 0.027), and median duration varies by only 1.2 minutes across a
63-year span (7.97 to 9.17 min by age band). Gender produces a small but directionally
consistent effect — 1.64 minutes among Customers and 1.05 among Subscribers — roughly a
quarter to a third the size of the user-type effect. This is a **negative result and is
reported as such**: the intuitive demographic explanations do not hold.

**5. Duration and volume move in opposite directions at weekends.** Customer median
duration rises from 12.03 to 14.88 minutes (+23.7%), while the Subscriber median is
effectively unchanged at 8.15 to 8.08. The Subscriber *mean* does rise, 9.72 to 10.49,
meaning a minority of Subscribers ride longer while the typical Subscriber trip is
identical — a tail effect, not a population shift. Consequently the two groups **diverge**
on duration at weekends, the median gap widening from 3.88 to 6.80 minutes.

Volume behaves oppositely. Normalised per day (February 2019 has 20 weekdays and 8 weekend
days), Subscriber trips fall from 6,887 to 3,137 — **45.5% of the weekday rate** — while
Customer trips barely move, 685 to 645, holding at **94.1%**. The Customer share of all
trips nearly doubles, from 9.0% on weekdays to 17.1% at weekends. Subscribers largely stop
riding; Customers ride at the same rate and ride longer.

## A Note on Analytical Choices

Two decisions materially affect the numbers above, and both work *against* the headline
finding rather than for it:

- **The 60-minute cap is not user-type neutral.** It retains 99.23% of Subscriber trips but
only 94.84% of Customer trips, because Customers ride longer. The Customer–Subscriber
median gap is 5.03 minutes uncapped and 4.50 capped, so every Customer figure here is a
**lower bound** and the true effect is about 11% larger. The cap is retained because
uncapped the Customer mean reaches 23.87 minutes against a median of 13.20 — distortion
from bikes left undocked for up to 23.7 hours, not from rides.

- **Missing demographics are retained, not dropped.** Birth year is absent for 8,265 trips
(5,028 Subscriber, 3,237 Customer). The *rate* differs sharply — 16.29% of Customer trips
against 3.07% of Subscriber trips, a 5.3x disparity — so filtering on age silently removes
one in six Customer trips. The age filter is therefore applied only in figures where age
is a plotted variable.